In [7]:
# ! pip install requests beautifulsoup4 pandas

In [8]:
import os
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import time


In [ ]:
def get_with_retry(url, retries=3, backoff=2):
    """GET a page with simple retries (the first run crashed around book 687).

    The encoding is forced to UTF-8: the first run let requests guess the
    encoding, which decoded the pages as ISO-8859-1 and produced mojibake
    such as 'Â£51.77' and 'nÃ´tre' in the CSV.
    """
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, timeout=10)
            response.encoding = "utf-8"
            return response
        except requests.RequestException:
            if attempt == retries:
                raise
            time.sleep(backoff * attempt)

def scrape_book_details(book_url, images_dir, book_id):
    """Scrapes individual book page for text, metadata, and downloads the image."""
    response = get_with_retry(book_url)
    if response.status_code != 200:
        return None
        
    soup = BeautifulSoup(response.text, "html.parser")
    
    # 1. EXTRACT RELEVANT DATA (For Classification)
    title_elem = soup.select_one('h1')
    title = title_elem.text.strip() if title_elem else None
    
    breadcrumb = soup.select('ul.breadcrumb li')
    category = breadcrumb[2].text.strip() if len(breadcrumb) > 2 else None
    
    description_elem = soup.select_one('#product_description ~ p')
    description = description_elem.text.strip() if description_elem else None
    
    # 2. DOWNLOAD IMAGE
    image_elem = soup.select_one('.item.active img')
    image_path = None
    if image_elem and 'src' in image_elem.attrs:
        image_url = urljoin(book_url, image_elem['src'])
        
        # Extract the extension and name it with the number (e.g., 1.jpg, 2.jpg)
        ext = os.path.splitext(image_url.split('/')[-1])[1]
        image_filename = f"{book_id}{ext}"
        image_path = os.path.join(images_dir, image_filename)
        
        # Only download if we haven't already
        if not os.path.exists(image_path):
            img_response = get_with_retry(image_url)
            if img_response.status_code == 200:
                with open(image_path, 'wb') as f:
                    f.write(img_response.content)
    
    # 3. EXTRACT IRRELEVANT DATA (Sandbox Noise)
    table_rows = soup.select('table.table-striped tr')
    table_data = {}
    for row in table_rows:
        th = row.select_one('th').text.strip()
        td = row.select_one('td').text.strip()
        table_data[th] = td
        
    return {
        # --- Relevant Features ---
        "Book_ID": book_id,
        "Title": title,
        "Description": description,
        "Category_Label": category,
        "Image_Local_Path": image_path,
        
        # --- Irrelevant Features ---
        "UPC": table_data.get("UPC"),
        "Price_Excl_Tax": table_data.get("Price (excl. tax)"),
        "Price_Incl_Tax": table_data.get("Price (incl. tax)"),
        "Tax": table_data.get("Tax"),
        "Availability": table_data.get("Availability"),
        "Number_of_Reviews": table_data.get("Number of reviews"),
        "Source_URL": book_url
    }

In [ ]:
def main():
    base_url = "https://books.toscrape.com/"
    catalogue_url = "https://books.toscrape.com/catalogue/page-{}.html"
    images_dir = "images"
    
    # Create images directory if it doesn't exist
    os.makedirs(images_dir, exist_ok=True)
    books_data = []
    book_counter = 1
    
    print("Starting scraper for books.toscrape.com...")
    
    # Iterate through all 50 pages (1000 books)
    for page_num in range(1, 51):
        print(f"Scraping page {page_num}/50...")
        url = catalogue_url.format(page_num)
        
        response = get_with_retry(url)
        if response.status_code != 200:
            print(f"Failed to fetch page {page_num}. Stopping pagination.")
            break
            
        soup = BeautifulSoup(response.text, "html.parser")
        book_links = soup.select('article.product_pod h3 a')
        
        # Loop through every book on the current page
        for link in book_links:
            # Construct the absolute URL for the book detail page
            book_url = urljoin(base_url + "catalogue/", link['href'])
            book_data = scrape_book_details(book_url, images_dir, book_counter)
            
            if book_data:
                books_data.append(book_data)
                book_counter += 1
                
        # Politeness delay to avoid hammering the server
        time.sleep(1)
        
    # Convert list of dictionaries to pandas DataFrame
    df = pd.DataFrame(books_data)
    
    # Reorder columns to separate relevant vs irrelevant clearly
    columns_order = [
        "Book_ID", "Title", "Description", "Category_Label", "Image_Local_Path", 
        "UPC", "Price_Excl_Tax", "Price_Incl_Tax", "Tax", "Availability", "Number_of_Reviews", "Source_URL"
    ]
    df = df[columns_order]
    
    # Save to CSV
    csv_filename = "books_dataset.csv"
    df.to_csv(csv_filename, index=False)
    
    print(f"\nScraping complete! Successfully scraped {len(df)} books.")
    print(f"Images saved to: ./{images_dir}/")
    print(f"Dataset saved to: {csv_filename}")
    

if __name__ == "__main__":
    main()